## PyINE prompt result database repackaging for Hugging Face

This notebook loads prompt result records from the framework's SQLite database and pushes them to a
HuggingFace dataset repository in Parquet format. Each row corresponds to a single LLM prompting
result, with structured columns for filtering and JSON columns for complex metadata fields.

**Prerequisites:** `pip install datasets huggingface_hub` and `huggingface-cli login`.

In [ ]:
import collections.abc
import json

import datasets
import huggingface_hub
import tqdm

import pyine.prompts
import pyine.utils.filesystem
import pyine.utils.reprod

pyine.utils.reprod.entrypoint_setup()

In [ ]:
# ------------ SETTINGS ------------
HF_REPO_NAME = "your-username/pyine-prompt-results"  # <-- change this
MAX_SHARD_SIZE = "1GB"

# optional filters (set to None to include everything)
TARGET_PROMPT_NAMES: list[str] | None = None  # e.g. ["hints/docs", "issues/todos"]
TARGET_PROMPT_VERSION: str | None = None
TAG_FILTER_RULE: str | None = None
# ----------------------------------

### Load prompt result database

In [ ]:
result_db = pyine.prompts.get_framework_db()
print(f"database path: {pyine.prompts.get_framework_db_path()}")
print(f"total records: {result_db.count_entries():,}")
print(f"prompt names: {result_db.list_prompt_names()}")
print(f"unique groups: {len(result_db.list_groups()):,}")
print(f"unique identifiers: {len(result_db.list_identifiers()):,}")

In [ ]:
# load records (optionally filtered by prompt name)
if TARGET_PROMPT_NAMES is not None:
    all_records: list[pyine.prompts.PromptResultRecord] = []
    for prompt_name in TARGET_PROMPT_NAMES:
        all_records.extend(
            result_db.get_by_prompt_name(
                prompt_name=prompt_name,
                prompt_version=TARGET_PROMPT_VERSION,
                tag_filter_rule=TAG_FILTER_RULE,
            )
        )
else:
    all_records = result_db.get_all_results(tag_filter_rule=TAG_FILTER_RULE)

print(f"loaded {len(all_records):,} records for export")

### Define record-to-row conversion

Each `PromptResultRecord` is flattened with:
- **Structured columns** for identity, prompt name/version, group, tags, creation metadata
- **Full text columns** for the prompt and LLM response
- **JSON string columns** for complex nested metadata (`meta`, `llm_params`, `llm_output`)

In [ ]:
def _safe_json_dumps(value: object) -> str:
    """Serializes a value to a JSON string, falling back to repr for non-serializable types."""
    if value is None:
        return "null"
    try:
        return json.dumps(value, ensure_ascii=False)
    except (TypeError, ValueError):
        return json.dumps(repr(value), ensure_ascii=False)


def record_to_hf_row(
    record: pyine.prompts.PromptResultRecord,
) -> dict:
    """Converts a PromptResultRecord into a flat dict suitable for HF datasets."""
    cmeta = record.creation_meta
    return {
        # ---- identity ----
        "record_uid": record.record_uid,
        "identifier": record.identifier,
        "prompt_name": record.prompt_name,
        "prompt_version": record.prompt_version,
        "group": record.group,
        "tags": record.tags,
        # ---- prompt + response (full text) ----
        "prompt": record.prompt,
        "result": record.result,
        # ---- creation metadata (flattened) ----
        "created_at": cmeta.created_at.isoformat(),
        "created_by": cmeta.created_by,
        "platform": cmeta.platform,
        "provider": cmeta.provider,
        # ---- complex metadata (JSON-serialized) ----
        "meta_json": _safe_json_dumps(record.meta),
        "llm_params_json": _safe_json_dumps(cmeta.llm_params),
        "llm_output_json": _safe_json_dumps(cmeta.llm_output),
    }

### Build HF dataset via generator

In [ ]:
def prompt_results_generator() -> collections.abc.Iterator[dict]:
    """Yields one HF-compatible dict per prompt result record."""
    for record in tqdm.tqdm(all_records, desc="repackaging prompt results"):
        yield record_to_hf_row(record)


hf_cache_dir = pyine.utils.filesystem.get_data_cache_subdir("hf_prompt_rpkg")
hf_dataset = datasets.Dataset.from_generator(
    prompt_results_generator,
    cache_dir=str(hf_cache_dir),
)
print(f"created HF dataset with {len(hf_dataset):,} rows and {len(hf_dataset.column_names)} columns")
print(f"columns: {hf_dataset.column_names}")

### Preview a sample row

In [ ]:
sample = hf_dataset[0]
for key, value in sample.items():
    if isinstance(value, str) and len(value) > 200:
        print(f"  {key}: {value[:200]}... ({len(value)} chars)")
    else:
        print(f"  {key}: {value}")

### Build dataset card and push to Hugging Face Hub

In [ ]:
# attach database-level summary metadata to the HF dataset info
prompt_names = result_db.list_prompt_names()
prompt_name_counts = result_db.count_entries(prompt_name=prompt_names, breakdown=True)
dataset_level_metadata = {
    "source_db_path": str(pyine.prompts.get_framework_db_path()),
    "total_records": len(all_records),
    "prompt_names": prompt_names,
    "prompt_name_counts": {name_tuple[0]: count for name_tuple, count in prompt_name_counts.items()},
    "unique_groups": len(result_db.list_groups()),
    "unique_identifiers": len(result_db.list_identifiers()),
}
if TARGET_PROMPT_NAMES is not None:
    dataset_level_metadata["filter_prompt_names"] = TARGET_PROMPT_NAMES
if TARGET_PROMPT_VERSION is not None:
    dataset_level_metadata["filter_prompt_version"] = TARGET_PROMPT_VERSION
if TAG_FILTER_RULE is not None:
    dataset_level_metadata["filter_tag_rule"] = TAG_FILTER_RULE

hf_dataset.info.description = (
    f"PyINE-v1 LLM prompt result database export. "
    f"Contains {len(all_records):,} records across {len(prompt_names)} prompt types."
)
hf_dataset.info.dataset_name = HF_REPO_NAME.split("/")[-1]
if hf_dataset.info.config_kwargs is None:
    hf_dataset.info.config_kwargs = {}
hf_dataset.info.config_kwargs["metadata"] = {
    key: str(val) if not isinstance(val, (str, int, float, bool, list)) else val
    for key, val in dataset_level_metadata.items()
}

print(f"dataset-level metadata: {json.dumps(dataset_level_metadata, indent=2, default=str)}")

In [ ]:
# build per-prompt-name breakdown for the card
prompt_name_counts_map = {name_tuple[0]: count for name_tuple, count in prompt_name_counts.items()}
prompt_table_rows = "\n".join(
    f"| `{name}` | {prompt_name_counts_map.get(name, 0):,} |" for name in sorted(prompt_name_counts_map.keys())
)

# collect unique providers and platforms from the records
providers = sorted({r.creation_meta.provider for r in all_records if r.creation_meta.provider})
platforms = sorted({r.creation_meta.platform for r in all_records})

filter_note = ""
if TARGET_PROMPT_NAMES or TARGET_PROMPT_VERSION or TAG_FILTER_RULE:
    filter_parts = []
    if TARGET_PROMPT_NAMES:
        filter_parts.append(f"prompt names: `{TARGET_PROMPT_NAMES}`")
    if TARGET_PROMPT_VERSION:
        filter_parts.append(f"prompt version: `{TARGET_PROMPT_VERSION}`")
    if TAG_FILTER_RULE:
        filter_parts.append(f"tag filter: `{TAG_FILTER_RULE}`")
    filter_note = (
        "\n> **Note:** This export was filtered to a subset of the full database: " + ", ".join(filter_parts) + ".\n"
    )


def _get_hf_size_category(count: int) -> str:
    """Returns the HuggingFace size category string for a given item count."""
    if count >= 1_000_000_000_000:
        return "n>1T"
    if count >= 1_000_000_000:
        return "1B<n<1T"
    if count >= 100_000_000:
        return "100M<n<1B"
    if count >= 10_000_000:
        return "10M<n<100M"
    if count >= 1_000_000:
        return "1M<n<10M"
    if count >= 100_000:
        return "100K<n<1M"
    if count >= 10_000:
        return "10K<n<100K"
    if count >= 1_000:
        return "1K<n<10K"
    return "n<1K"


card_data = huggingface_hub.DatasetCardData(
    language="en",
    license="cc-by-4.0",
    tags=["code", "code-augmentation", "llm-annotations", "python", "code-analysis", "prompt-results"],
    task_categories=["text-generation", "code-execution"],
    size_categories=[_get_hf_size_category(len(all_records))],
    pretty_name="PyINE-v1 LLM Prompt Results",
)

card_content = f"""\
# PyINE-v1 LLM Prompt Results

This dataset contains **{len(all_records):,}** LLM prompting results generated by the
[PyINE](https://github.com/TODO/pyine) framework. Each row is a single prompt/response
pair produced by querying an LLM to annotate or augment Python code solutions (e.g.
adding documentation hints, injecting bugs, generating misleading comments).
{filter_note}
## Dataset structure

### Prompt types

| Prompt name | Records |
|-------------|---------|
{prompt_table_rows}

### Columns

**Identity:**
| Column | Type | Description |
|--------|------|-------------|
| `record_uid` | `string` | Unique record ID (content-hashed) |
| `identifier` | `string` | Record identifier (typically a trace or solution ID) |
| `prompt_name` | `string?` | Name of the prompt template used |
| `prompt_version` | `string?` | Version of the prompt template |
| `group` | `string?` | Optional grouping label |
| `tags` | `list[string]` | Tags for filtering (e.g. `augment:*`, `created_by:*`) |

**Prompt and response:**
| Column | Type | Description |
|--------|------|-------------|
| `prompt` | `string` | Full prompt text sent to the LLM |
| `result` | `string` | Full LLM response text |

**Creation metadata:**
| Column | Type | Description |
|--------|------|-------------|
| `created_at` | `string` | ISO 8601 UTC timestamp |
| `created_by` | `string` | Username/identifier of the creator |
| `platform` | `string` | Platform where the record was created |
| `provider` | `string?` | LLM provider/service name |

**Serialized metadata (JSON strings):**
| Column | Type | Description |
|--------|------|-------------|
| `meta_json` | `string` | Additional record metadata |
| `llm_params_json` | `string` | LLM hyperparameters (model, temperature, etc.) |
| `llm_output_json` | `string` | Provider-specific output metadata (token counts, etc.) |

### Providers and platforms

- **LLM providers:** {", ".join(f"`{p}`" for p in providers) if providers else "_not recorded_"}
- **Platforms:** {", ".join(f"`{p}`" for p in platforms) if platforms else "_not recorded_"}

## Usage

```python
import datasets

ds = datasets.load_dataset("{HF_REPO_NAME}")

# filter by prompt type
hints_ds = ds.filter(lambda row: row["prompt_name"] == "hints/docs")

# access a record
record = ds[0]
print(f"prompt name: {{record['prompt_name']}}")
print(f"response length: {{len(record['result'])}} chars")
```

## Source

Generated by the [PyINE](https://github.com/TODO/pyine) framework's LLM annotation pipeline.
"""

dataset_card = huggingface_hub.DatasetCard(card_content)
dataset_card.data = card_data
print(str(dataset_card)[:2000] + "\n...")

In [ ]:
hf_dataset.push_to_hub(
    repo_id=HF_REPO_NAME,
    max_shard_size=MAX_SHARD_SIZE,
)
dataset_card.push_to_hub(HF_REPO_NAME, repo_type="dataset")
print(f"pushed to https://huggingface.co/datasets/{HF_REPO_NAME}")